In [4]:
%reset -f

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import joblib
import json
from pathlib import Path
import copy

#BASE_DIR = Path(__file__).resolve().parent
import sys
#sys.path.append("/work/home/acychkqsp6/zrm/MasterThesis/tools")
sys.path.append("../../tools")
import architec as acht
import evaluation as eva
import prepare as prep

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
import importlib
#importlib.reload(eva)


In [6]:
NumSpecies = 20
featgam = NumSpecies**2 - NumSpecies
task_num = 3+NumSpecies # 0: num. of stable states, 1: stable-state entropy, 2: diversity, 3: abundance, 3+NumSpecies: switching species, 3+2*NumSpecies: oscillation

# The folder storing datasets splited latter.
#folder_path = "/work/home/acychkqsp6/zrm/MasterThesis/Databases/Ver1.9/ver1.9.6_upto20"
folder_path = "../../Databases/Ver1.9/ver1.9.6_upto20"

for rep in range(1,16):
    #split_df = pd.read_csv(f"/work/home/acychkqsp6/zrm/MasterThesis/Principle_design_SynCom/Fig2/split_MD_v1/split_MD_{rep}.csv")
    split_df = pd.read_csv(f"split_MD_v1/split_MD_{rep}.csv")

    #Trainset
    narray_tr = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="tr"
    )

    #Testset
    narray_te = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="te"
    )

    #-------------------------------------------------------
    Xg_tr, XA_tr, Xd_tr, y_tr, scaler_g, scaler_d, idx_tr = prep.prepare_train(
        narray_tr,
        NumSpecies,
        featgam,
        task_num,
        #sample_size=97000,
        random_state=42
    )

    Xg_te, XA_te, Xd_te, y_te, idx_te = prep.prepare_test(
        narray_te,
        NumSpecies,
        featgam,
        scaler_g,
        scaler_d,
        task_num,
        #sample_size=20000,
        random_state=42
    )

    train_dataset = acht.CommunityDataset_MultiLabel(Xg_tr, XA_tr, Xd_tr, y_tr)
    test_dataset = acht.CommunityDataset_MultiLabel(Xg_te, XA_te, Xd_te, y_te)

    n_val = int(len(train_dataset)*0.2)
    n_train = len(train_dataset)-n_val
    train_ds, val_ds = random_split(train_dataset, [n_train,n_val])

    train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

    model = acht.SSTrans2(NumSpecies).to(device) # Select deep learning architecture.
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    early_stop = acht.EarlyStopping(patience=8, min_delta=1e-4)
    history = {"loss":[], "val_loss":[]}

    best_state = None

    for epoch in range(140):
        model.train()
        train_loss = 0
        
        for Xg,XA,Xd,y in train_loader:

            Xg = Xg.to(device)
            XA = XA.to(device)
            Xd = Xd.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(Xg,XA,Xd)

            loss = criterion(logits,y)
            loss.backward()

            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        model.eval()

        val_loss = 0

        with torch.no_grad():

            for Xg,XA,Xd,y in val_loader:

                Xg = Xg.to(device)
                XA = XA.to(device)
                Xd = Xd.to(device)
                y = y.to(device)

                logits = model(Xg,XA,Xd)
                loss = criterion(logits,y)
                val_loss += loss.item()

        val_loss /= len(val_loader)

        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch+1} "
            f"train={train_loss:.5f} "
            f"val={val_loss:.5f}"
        )

        if val_loss < early_stop.best_loss:
            #best_state = model.state_dict()
            best_state = copy.deepcopy(model.state_dict())

        if early_stop.step(val_loss):
            print("Early stopping")
            break

    model.load_state_dict(best_state)

    # Model saving
    #save_dir = Path(f"/work/home/acychkqsp6/zrm/MasterThesis/saved_models/swi_species_15fold/sp3to20_SS_transpt3_r{rep}")
    #save_dir = Path(f"../../saved_models/swi_species_15fold/sp3to20_SS_transpt3_r{rep}")
    save_dir = Path(f"../../saved_models/swi_species_15fold/sp10_SS_transpt3_r{rep}")

    save_dir.mkdir(parents=True, exist_ok=True)  # auto-create directory
    torch.save(model.state_dict(), save_dir / "model.pt")
    joblib.dump(scaler_g, save_dir / "scaler_g.pkl")
    joblib.dump(scaler_d, save_dir / "scaler_d.pkl")

    config = {
        "NumSpecies": NumSpecies,
        "featgam": featgam,
        "task_num": task_num,
        "embed_dim": 64,
        "num_heads": 4,
        "ff_dim": 128
    }

    with open(save_dir / "config.json", "w") as f:
        json.dump(config, f)


tr shape = (73779, 465)
te shape = (18444, 465)
Epoch 1 train=0.30494 val=0.26879
Epoch 2 train=0.26383 val=0.25750
Epoch 3 train=0.25543 val=0.25101
Epoch 4 train=0.25100 val=0.25191
Epoch 5 train=0.24860 val=0.24695
Epoch 6 train=0.24599 val=0.24311
Epoch 7 train=0.24378 val=0.24310
Epoch 8 train=0.24182 val=0.23963
Epoch 9 train=0.23985 val=0.23798
Epoch 10 train=0.23856 val=0.23619
Epoch 11 train=0.23710 val=0.23687
Epoch 12 train=0.23567 val=0.23728
Epoch 13 train=0.23489 val=0.23435
Epoch 14 train=0.23300 val=0.23287
Epoch 15 train=0.23250 val=0.23144
Epoch 16 train=0.23153 val=0.23135
Epoch 17 train=0.23022 val=0.23074
Epoch 18 train=0.22931 val=0.22890
Epoch 19 train=0.22820 val=0.22819
Epoch 20 train=0.22784 val=0.22826
Epoch 21 train=0.22750 val=0.22783
Epoch 22 train=0.22653 val=0.22686
Epoch 23 train=0.22615 val=0.22630
Epoch 24 train=0.22536 val=0.22685
Epoch 25 train=0.22469 val=0.22654
Epoch 26 train=0.22435 val=0.22534
Epoch 27 train=0.22393 val=0.22534
Epoch 28 train=0

KeyboardInterrupt: 